# 対話状態の追跡を評価する（JMultiWOZ、全スロット一致率）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_dialogue/dst_eval.ipynb)

NLP クックブック 26-14 ／ 対話システム

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

タスク指向対話のシステムは、利用者がここまでに伝えた条件（都市、店のジャンル、予約の人数や日時）を対話状態として持ち続ける。この章の「必要な情報を聞き出しながら予約を受け付ける（スロットフィリング）」では規則で取り出したが、実際の対話データで、LLM が各ターンの対話状態をどれだけ正しく追跡できるかを、全スロット一致率 (JGA) とスロット単位の正解率で測りたい。

In [1]:
import collections
import json
import unicodedata
import urllib.request
import zipfile

# JMultiWOZ（名古屋大学ほか、CC BY-SA 4.0）: 旅行案内の 6 ドメイン（飲食店、宿泊施設、観光名所、買い物、タクシー、天気）の日本語の対話 4,246 件
urllib.request.urlretrieve("https://github.com/nu-dialogue/jmultiwoz/raw/master/dataset/JMultiWOZ_1.0.zip", "JMultiWOZ_1.0.zip")   # 約 11MB
with zipfile.ZipFile("JMultiWOZ_1.0.zip") as z:
    dialogues, split, ontology = (json.loads(z.read(f"JMultiWOZ_1.0/{f}.json")) for f in ["dialogues", "split_list", "ontology"])

# スロットの一覧（「飲食店-ジャンル」のような名前にする）。着目しているドメイン (active_domain) は利用者の条件ではないので除く
SLOTS = {(dom, s): f"{v['ja']}-{x['ja']}" for dom, v in ontology.items() for s, x in v["slots"].items()
         if (x["informable"] or x["book"]) and s != "active_domain"}

def state_of(turn):
    """SYSTEM のターンに付いた対話状態（belief_state と予約の book_state）から、値のあるスロットだけを取り出す"""
    st = turn["dialogue_state"]
    return {SLOTS[dom, s]: v for part in ["belief_state", "book_state"] for dom, slots in st[part].items()
            for s, v in slots.items() if (dom, s) in SLOTS and v is not None}

def examples(names):
    """利用者の発話ごとに、直前の案内係の発話、1 つ前の利用者のターンの正解の状態、このターンの正解の状態を組にする"""
    out = []
    for n in names:
        turns, prev = dialogues[n]["turns"], {}
        for i, t in enumerate(turns[:-1]):
            if t["speaker"] == "USER":
                gold = state_of(turns[i + 1])
                out.append({"dialogue": n, "turn": i // 2, "system": turns[i - 1]["utterance"] if i else "",
                            "user": t["utterance"], "prev": prev, "gold": gold})
                prev = gold
    return out

train = examples(split["train"])
# 値の種類が少ないスロット（値段帯、wifi の有無など）は、学習データに出る値を候補としてプロンプトに書く
values = collections.defaultdict(collections.Counter)
for x in train:
    for k, v in x["gold"].items():
        values[k][v] += 1
CHOICES = {k: sorted(values[k]) for k in SLOTS.values() if 0 < len(values[k]) <= 6 and not k.endswith(("人数", "日数"))}

test = examples(split["test"][:20])                    # 評価用 300 対話のうち 20 対話に絞る
print(len(SLOTS), "スロット", len(test), "ターン（20 対話）")
for x in test[:4]:
    print(f"案内係: {x['system']}\n利用者: {x['user']}\n  → {x['gold']}")

42 スロット 151 ターン（20 対話）
案内係: 
利用者: 大阪で宿泊施設を探しているのですが。
  → {'一般-都市': '大阪'}
案内係: ご利用ありがとうございます。エリアなどでご希望はございませんか？
利用者: レストラン併設で安めのところがいいです。
  → {'一般-都市': '大阪', '宿泊施設-値段帯': '安め', '宿泊施設-レストランの有無': '有り'}
案内係: 申し訳ございません。その条件では、該当する施設が見つかりませんでした。
利用者: そうですか。じゃあ、レストランのないところでもいいです。
  → {'一般-都市': '大阪', '宿泊施設-値段帯': '安め'}
案内係: ありがとうございます。では、ホテル東洋はいかがでしょうか？
利用者: いいですね。チェックアウト終了時刻は何時でしょうか。
  → {'一般-都市': '大阪', '宿泊施設-施設名': 'ホテル東洋', '宿泊施設-値段帯': '安め'}


In [2]:
import re
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen3-4B-Instruct-2507（Apache 2.0）。T4 では float16 で約 8GB。この章のほかのレシピの sarashina2.2-3b より、JSON の出力と指示に従うのが安定していた
name = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

INSTRUCTION = ("旅行案内の対話で、利用者の最新の発話によって新しく決まった条件や、変わった条件を JSON で書いてください。"
               "「これまでの状態」に既にあって変わらない項目は書かず、何も変わらなければ {} と書いてください。\n"
               f"項目の名前: {'、'.join(SLOTS.values())}\n"
               "次の項目は、値をこの中から 1 つ選ぶ: " + "、".join(f"{k}（{'、'.join(v)}）" for k, v in CHOICES.items()) + "\n"
               "日付は 1/17 のような「月/日」、時刻は 09:40 のような 2 桁の 24 時間制、人数と宿泊日数は 2 のような数字だけで書く。")
# データの付け方の約束。プロンプトの最後に置く（冒頭に置いたときは、都市や施設名をよく書き漏らした）
RULES = ("# 注意\n"
         "- 旅行先の都市が出てきたら、一般-都市 に書く。\n"
         "- 案内係が勧めた施設を利用者が受け入れたり、その施設について質問したりしたら、その名前を案内係の発話のとおりに"
         "名前の項目（店名、施設名、名称、店舗名、タクシー会社名）に書く。\n"
         "- 利用者が条件を取り消したら（「〜でなくてもいい」）、その項目の値を null にする。")

def prompt(prev, system, user):
    return (f"{INSTRUCTION}\n\n# これまでの状態\n{json.dumps(prev, ensure_ascii=False)}\n\n"
            f"# 案内係の直前の発話\n{system or '(なし)'}\n\n# 利用者の最新の発話\n{user}\n\n{RULES}")

def update_of(x):
    """正解の状態の差分: 1 つ前のターンから増えた・変わった項目と、消えた項目 (null)"""
    return {**{k: v for k, v in x["gold"].items() if x["prev"].get(k) != v}, **{k: None for k in x["prev"] if k not in x["gold"]}}

# 学習データから、都市・施設名の受け入れ・条件の取り消し・予約の日時の 4 例を選んで見せる（少数例 = few-shot）
wants = [lambda x, u: x["turn"] == 0 and "一般-都市" in u and len(u) >= 2,
         lambda x, u: any(k.endswith(("店名", "施設名", "名称")) and v and v not in x["user"] for k, v in u.items()),
         lambda x, u: None in u.values(),
         lambda x, u: any("日付" in k for k in u) and any("時刻" in k for k in u)]
SHOTS = []
for want in wants:
    x = next(x for x in train if want(x, update_of(x)))
    SHOTS += [{"role": "user", "content": prompt(x["prev"], x["system"], x["user"])},
              {"role": "assistant", "content": json.dumps(update_of(x), ensure_ascii=False)}]

def parse(text):
    """出力から最初の {...} を取り出して JSON として読む。読めなければ「変化なし」とする"""
    m = re.search(r"\{.*?\}", text, re.S)
    try:
        d = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        return {}
    return {k: None if v in (None, "", "null") else str(v) for k, v in d.items() if k in SLOTS.values()}

@torch.no_grad()
def generate(prompts, bs=4):
    out = []
    for k in range(0, len(prompts), bs):
        msgs = [SHOTS + [{"role": "user", "content": p}] for p in prompts[k:k + bs]]
        enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True, padding=True).to(llm.device)
        o = llm.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        out += tokenizer.batch_decode(o[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return out

def track(xs, oracle=False):
    """ターンごとに状態の差分を推定し、前の状態に足していく。oracle=True なら、前の状態に正解を使う（そのターンだけの誤りを見る）"""
    by_dialogue = collections.defaultdict(list)
    for x in xs:
        by_dialogue[x["dialogue"]].append(x)
    state = {d: {} for d in by_dialogue}
    preds = {}
    for t in range(max(len(v) for v in by_dialogue.values())):          # 同じターン目を対話をまたいでまとめて生成する
        items = [v[t] for v in by_dialogue.values() if t < len(v)]
        prevs = [x["prev"] if oracle else state[x["dialogue"]] for x in items]
        for x, prev, out in zip(items, prevs, generate([prompt(p, x["system"], x["user"]) for x, p in zip(items, prevs)])):
            new = {k: v for k, v in {**prev, **parse(out)}.items() if v is not None}
            preds[id(x)] = state[x["dialogue"]] = new
    return [preds[id(x)] for x in xs]

t = time.time()
pred = track(test)                      # 自分の推定を引き継ぐ（実際の使い方）
print(f"推定した状態を引き継ぐ: {time.time() - t:.0f} 秒")
t = time.time()
pred_oracle = track(test, oracle=True)  # 前のターンの正解から始める
print(f"前のターンの正解から: {time.time() - t:.0f} 秒")
print("GPU メモリの最大使用量:", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")

推定した状態を引き継ぐ: 361 秒
前のターンの正解から: 361 秒
GPU メモリの最大使用量: 13.4 GB


In [3]:
import pandas as pd

def norm(state):
    """全角半角と空白の違いは無視して比べる"""
    return {k: unicodedata.normalize("NFKC", v).replace(" ", "").lower() for k, v in state.items()}

def scores(preds):
    golds, preds = [norm(x["gold"]) for x in test], [norm(p) for p in preds]
    jga = sum(g == p for g, p in zip(golds, preds)) / len(test)                     # 全スロットが一致したターンの割合
    slot_acc = sum(g.get(s) == p.get(s) for g, p in zip(golds, preds) for s in SLOTS.values()) / (len(test) * len(SLOTS))
    tp = sum(g.get(s) == v for g, p in zip(golds, preds) for s, v in p.items())
    prec, rec = tp / max(1, sum(map(len, preds))), tp / sum(map(len, golds))
    return {"JGA": jga, "スロット正解率": slot_acc, "適合率": prec, "再現率": rec}

print(pd.DataFrame({"何も埋めない": scores([{} for _ in test]),
                    "1 つ前のターンの正解を写す（参考）": scores([x["prev"] for x in test]),
                    "LLM: 推定した状態を引き継ぐ": scores(pred),
                    "LLM: 前のターンの正解から（参考）": scores(pred_oracle)}).T.round(3))

# ターンが進むほど状態は大きくなり、全スロット一致は難しくなる
df = pd.DataFrame({"ターン": [min(x["turn"], 8) for x in test], "スロット数": [len(x["gold"]) for x in test],
                   "一致": [norm(x["gold"]) == norm(p) for x, p in zip(test, pred)]})
print(df.groupby("ターン").agg(件数=("一致", "size"), 正解のスロット数=("スロット数", "mean"), JGA=("一致", "mean")).round(2).T)

                       JGA  スロット正解率    適合率    再現率
何も埋めない               0.000    0.844  0.000  0.000
1 つ前のターンの正解を写す（参考）   0.172    0.962  0.975  0.773
LLM: 推定した状態を引き継ぐ     0.258    0.948  0.949  0.693
LLM: 前のターンの正解から（参考）  0.589    0.985  0.983  0.920
ターン          0      1      2      3      4      5      6      7      8
件数        20.0  20.00  20.00  19.00  19.00  16.00  14.00   9.00  14.00
正解のスロット数   2.2   3.40   4.60   5.37   7.37   9.31  10.36  11.22  10.71
JGA        0.9   0.55   0.35   0.11   0.05   0.00   0.00   0.00   0.00


In [4]:
# スロットごとに: 正解に値があるターンで値を当てた割合（再現率）と、正解にない値を書いた回数
rows = []
for s in SLOTS.values():
    g = [(norm(x["gold"]).get(s), norm(p).get(s)) for x, p in zip(test, pred)]
    with_gold = [(a, b) for a, b in g if a is not None]
    if with_gold:
        rows.append({"スロット": s, "正解に値がある": len(with_gold), "再現率": sum(a == b for a, b in with_gold) / len(with_gold),
                     "余計な値": sum(a is None and b is not None for a, b in g)})
print(pd.DataFrame(rows).sort_values("正解に値がある", ascending=False).head(12).round(2).to_string(index=False))

# 誤りの例: 前のターンの正解から始めても間違えたターンについて、違うスロットだけを並べる
for x, p in [(x, p) for x, p in zip(test, pred_oracle) if norm(x["gold"]) != norm(p)][:5]:
    g, q = norm(x["gold"]), norm(p)
    print(f"\n案内係: {x['system'][:50]}\n利用者: {x['user'][:50]}")
    print("  違い（正解, 予測）:", {s: (g.get(s), q.get(s)) for s in sorted(set(g) | set(q)) if g.get(s) != q.get(s)})

         スロット  正解に値がある  再現率  余計な値
        一般-都市      151 1.00     0
     宿泊施設-施設名       53 0.11     0
       飲食店-店名       47 0.21     2
      観光名所-名称       44 0.20     0
      飲食店-値段帯       43 1.00     0
     飲食店-ジャンル       41 0.71     0
 宿泊施設-wifiの有無       39 1.00     0
     宿泊施設-値段帯       36 0.81     0
  観光名所-駐車場の有無       35 1.00     0
      飲食店-エリア       25 0.48     8
宿泊施設-レストランの有無       23 0.74     0
 観光名所-wifiの有無       22 0.73     1

案内係: ありがとうございます。では、ホテル東洋はいかがでしょうか？
利用者: いいですね。チェックアウト終了時刻は何時でしょうか。
  違い（正解, 予測）: {'宿泊施設-施設名': ('ホテル東洋', None)}

案内係: かしこまりました。日本タクシー株式会社はいかがですか？
利用者: ジャンボタクシーには対応していますか？
  違い（正解, 予測）: {'タクシー-ジャンボタクシー対応': (None, '対応')}

案内係: はい、ご利用いただけます。
利用者: では、8/16の12:00に予約をお願いします。
  違い（正解, 予測）: {'タクシー-ジャンボタクシー対応': ('対応', None), 'タクシー-出発時刻': ('12:00', None), 'タクシー-日付': ('8/16', None), '宿泊施設-日付': (None, '8/16')}

案内係: かしこまりました。では、鶴見緑地湯元 水春はいかがでしょうか？
利用者: ありがとうございます。見緑地湯元 水春の料金情報，子供入場料は分かりますか？
  違い（正解, 予測）: {'観光名所-名称': ('鶴見緑地湯元水春', None)}

案内係: かしこまりました。では、MARUZEN&ジュン

- データは JMultiWOZ（Ohashi ら, LREC-COLING 2024、CC BY-SA 4.0）。英語の MultiWOZ と同じく、利用者役と案内係役の 2 人の作業者が対話し、案内係のターンごとに対話状態（スロットと値）が付いている。ここでは評価用の 20 対話（151 ターン、42 スロット）に絞った。
- JGA（Joint Goal Accuracy）は、そのターンのすべてのスロットの値が正解と一致した割合で、1 つでも違えば 0 になる。LLM が自分の推定を引き継ぐと 0.258 だった。ターンが進んで正解のスロットが増えるほど下がり、最初のターンは 0.90、4 ターン目は 0.11、6 ターン目以降は 0 だった。一度間違えた値は、直さない限り後のターンにずっと残る。
- スロット正解率は、値のないスロットも「値なし」が一致すれば正解に数えるので、高く出る。何も埋めなくても 0.844、LLM は 0.948 で、1 つ前のターンの正解を写すだけのほう (0.962) が高い。値のあるスロットの適合率と再現率（LLM は 0.949 と 0.693）や JGA と並べて見る。
- 誤りがどこから来るかを分けるため、前のターンの状態に正解を与えて、そのターンの更新だけを推定させた（`oracle=True`）。JGA は 0.589 で、1 ターンごとの誤りは少なくても、それが積み重なると大きく下がることが分かる。対話全体を毎回読み直して状態をまるごと出力させる方法も試したが、sarashina2.2-3b-instruct では出力の 4 分の 1 ほどが JSON として読めず、JGA は 0.04 だった。
- 一番落とすのは施設の名前（再現率 0.11〜0.21）。正解のデータでは、案内係が勧めた施設を利用者が受け入れたり、その施設について尋ねたりした時点で名前が状態に入るが、利用者は「いいですね」「そこの最寄り駅は」としか言わない。注意書きに書いても多くは拾えず、本屋を飲食店の店名に入れるようなドメインの取り違えもあった。逆に、利用者が「ジャンボタクシーには対応していますか」と尋ねただけでは、正解の状態には入らない。どこまでを状態とするかはデータの付け方の約束で決まり、プロンプトでそれを伝えきるのは難しい。
- 都市の書き漏らしは、データの付け方の約束（注意書き）をプロンプトの最後に移したら無くなった。別に試した 15 対話では、冒頭に置いたときの JGA は最後に置いたときの約半分だった。約束を書き足す前の同じ方法で比べると、sarashina2.2-3b-instruct の JGA (0.035) は Qwen3-4B (0.088) より低かった。
- 実用の DST は、学習データでモデルを fine-tuning することが多い。JMultiWOZ の論文（評価用の全 300 対話）では、fine-tuning した T5-large の JGA が 0.77、少数例を与えた GPT-4 が 0.36 と報告されている（スロットの範囲や比べ方がこのレシピとは違うので、数値はそのまま比べられない）。
- 2 回の推定は検証の GPU でそれぞれ約 6 分かかった。T4 ではさらに数倍かかる。GPU メモリの最大使用量は 13.4GB で、T4 の 16GB に近い。足りなければ `generate` の `bs` を 2 にするか、対話数を減らす。モデルは Apache 2.0 ライセンス。